# Coloração de Vértice

Objetivo: Implementar algoritmo de força bruta e uma heurística para o problema de Coloração de Vértice
em Grafos, que consiste em encontrar o número cromático $\chi(G)$ de um grafo 

O trabalho é composto de duas partes, aplicação do algoritmo de força-bruta em instâncias do
problema de coloração de vértices e aplicação de uma heurística em cinco instâncias do mesmo problema.
O problema de Coloração de Vértice consiste em, dado um grafo qualquer, dois vértices adjacentes
não devem receber a mesma cor. O número cromático 𝜒(𝐺) de um grafo 𝐺 é o menor número de cores
necessárias para colorir os vértices de um grafo de modo que vértices adjacentes não tenham a mesma cor.
Se o número de cores utilizado na coloração de vértices de um grafo for igual a 𝜒(𝐺), a coloração é dita ótima.

O problema de **Coloração de Vértices** é um dos problemas clássicos da teoria dos grafos e da ciência da computação. Dado um grafo $G = (V, E)$, o objetivo é atribuir cores aos vértices de modo que:

1. Vértices adjacentes tenham cores diferentes
2. O número total de cores seja **minimizado**

Formalmente:

$$\chi(G) = \min\{k : \exists \ f: V \rightarrow \{1, 2, ..., k\} \ tal \ que \ (u,v) \in E \Rightarrow f(u) \neq f(v)\}$$

### Aplicações Práticas

O problema de coloração de vértices possui diversas aplicações práticas:

- **Alocação de Frequências em Rádios**: Estações de rádio próximas geograficamente não podem usar a mesma frequência
- **Planejamento de Horários**: Disciplinas compartilham o mesmo horário apenas se seus alunos não se sobrepõem
- **Atribuição de Registros em Compiladores**: Variáveis que são usadas simultaneamente não podem compartilhar o mesmo registrador
- **Coloração de Mapas Geográficos**: Regiões adjacentes devem ter cores diferentes

### Complexidade Computacional

O problema de coloração de vértices é **NP-Completo**, o que significa que:

- Não existe algoritmo conhecido que o resolva em tempo polinomial para o caso geral
- A complexidade cresce exponencialmente com o tamanho do grafo
- Para instâncias grandes, é necessário usar heurísticas e metaheurísticas

---

## Parte 1: Algoritmo de Força Bruta

1. Implementar o método de força bruta para solucionar o problema, ou seja, um algoritmo que
determina todas as combinações de cores a melhor, ou seja, a menor;
2. Gerar instâncias de tamanho 2 à n e aplicar o método implementado no item 1.
3. Computar o tempo de execução durante a aplicação da força-bruta em cada uma das instâncias
geradas. A aplicação do método deve ser realizada em quantas instâncias forem possíveis
(possivelmente o tamanho máximo vai girar em torno de 13 a 20 cidades);
Obs.: AS instâncias devem ser geradas de maneira automática e aleatória. Os grafos devem ser não
direcionados e não ponderados. Não deve haver loops nem arestas paralelas nas instâncias geradas.
Pode-se utilizar qualquer tipo de representação de grados que se desejar.


O algoritmo de força bruta tenta todas as possíveis combinações de colorações até encontrar a melhor solução ótima.

### Estratégia

1. Iniciar com $k = 1$ cores
2. Tentar todas as combinações de coloração usando $k$ cores
3. Se encontrar uma coloração válida, retornar $k$ como o número cromático
4. Caso contrário, incrementar $k$ e repetir

### Complexidade

Para um grafo com $n$ vértices:
- No pior caso: $O(k^n)$ onde $k$ é o número cromático
- Cresce exponencialmente com o tamanho do grafo
- Viável para grafos com até ~15-20 vértices

In [1]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import time
from itertools import product
import pandas as pd
import seaborn as sns
from collections import defaultdict

# Configurações
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Bibliotecas importadas com sucesso!")

ModuleNotFoundError: No module named 'networkx'

In [ ]:
def eh_coloracao_valida(grafo, coloracao):
    """
    Verifica se uma coloração é válida para um grafo.
    Uma coloração é válida se vértices adjacentes têm cores diferentes.
    
    Args:
        grafo: Grafo NetworkX
        coloracao: Dicionário {vértice: cor}
    
    Returns:
        True se a coloração é válida, False caso contrário
    """
    for u, v in grafo.edges():
        if coloracao[u] == coloracao[v]:
            return False
    return True


def numero_cromatico_forca_bruta(grafo):
    """
    Encontra o número cromático usando força bruta.
    Tenta todas as possíveis combinações de cores.
    
    Args:
        grafo: Grafo NetworkX
    
    Returns:
        (número cromático, coloração ótima)
    """
    vertices = list(grafo.nodes())
    n = len(vertices)
    
    # Tentar com k cores, começando com k=1
    for k in range(1, n + 1):
        # Gerar todas as combinações possíveis de cores
        for coloracao_tuple in product(range(k), repeat=n):
            coloracao = {vertices[i]: coloracao_tuple[i] for i in range(n)}
            
            if eh_coloracao_valida(grafo, coloracao):
                return k, coloracao
    
    # Se nenhuma coloração foi encontrada (não deve acontecer)
    return n, {v: i for i, v in enumerate(vertices)}


print("Funções de força bruta definidas.")

### Geração Automática de Instâncias Aleatórias

Vamos gerar grafos aleatórios sem loops nem arestas paralelas, variando o tamanho de 2 a n vértices.

In [ ]:
def gerar_grafo_aleatorio(n_vertices, probabilidade=0.3):
    """
    Gera um grafo aleatório não direcionado sem loops nem arestas paralelas.
    
    Args:
        n_vertices: Número de vértices
        probabilidade: Probabilidade de existir uma aresta entre dois vértices
    
    Returns:
        Grafo NetworkX
    """
    G = nx.Graph()
    G.add_nodes_from(range(n_vertices))
    
    # Adicionar arestas aleatoriamente
    for i in range(n_vertices):
        for j in range(i + 1, n_vertices):
            if np.random.random() < probabilidade:
                G.add_edge(i, j)
    
    return G


def testar_forca_bruta_escalabilidade(tamanhos=[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], 
                                       probabilidade=0.3, 
                                       repeticoes=3):
    """
    Testa o algoritmo de força bruta em grafos de tamanhos crescentes.
    Mede o tempo de execução para cada tamanho.
    
    Args:
        tamanhos: Lista de tamanhos de grafos a testar
        probabilidade: Probabilidade de aresta
        repeticoes: Número de repetições para cada tamanho
    
    Returns:
        DataFrame com resultados
    """
    resultados = []
    
    for n in tamanhos:
        tempos = []
        cromaticos = []
        
        print(f"Testando tamanho {n}...", end=" ")
        
        for _ in range(repeticoes):
            G = gerar_grafo_aleatorio(n, probabilidade)
            
            start = time.time()
            chi, _ = numero_cromatico_forca_bruta(G)
            elapsed = time.time() - start
            
            tempos.append(elapsed)
            cromaticos.append(chi)
        
        tempo_medio = np.mean(tempos)
        chi_medio = np.mean(cromaticos)
        
        resultados.append({
            'Tamanho': n,
            'Tempo Médio (s)': tempo_medio,
            'Número Cromático Médio': chi_medio
        })
        
        print(f"χ={chi_medio:.1f}, tempo={tempo_medio:.4f}s")
    
    return pd.DataFrame(resultados)


print("Funções de geração de instâncias definidas.")

### Execução e Análise de Escalabilidade

Vamos testar o algoritmo de força bruta em grafos de tamanhos crescentes e visualizar o crescimento exponencial do tempo.

In [ ]:
# Executar testes de escalabilidade
# AVISO: Grafos com mais de 13-15 vértices podem levar muito tempo!

tamanhos_teste = list(range(2, 13))  # De 2 a 12 vértices
resultados_forca_bruta = testar_forca_bruta_escalabilidade(
    tamanhos=tamanhos_teste,
    probabilidade=0.3,
    repeticoes=2
)

print("\n=== Resultados da Força Bruta ===")
print(resultados_forca_bruta)

In [ ]:
# Visualizar crescimento exponencial
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Tempo vs Tamanho (escala linear)
axes[0].plot(resultados_forca_bruta['Tamanho'], resultados_forca_bruta['Tempo Médio (s)'], 
             'o-', linewidth=2, markersize=8, color='steelblue')
axes[0].set_xlabel('Número de Vértices', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Tempo de Execução (segundos)', fontsize=12, fontweight='bold')
axes[0].set_title('Força Bruta: Crescimento do Tempo (Escala Linear)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Tempo vs Tamanho (escala logarítmica)
axes[1].semilogy(resultados_forca_bruta['Tamanho'], resultados_forca_bruta['Tempo Médio (s)'], 
                  'o-', linewidth=2, markersize=8, color='darkred')
axes[1].set_xlabel('Número de Vértices', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Tempo de Execução (segundos - log)', fontsize=12, fontweight='bold')
axes[1].set_title('Força Bruta: Crescimento Exponencial (Escala Log)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, which='both')

plt.tight_layout()

# Salvar gráfico
caminho_relatorio = os.path.join('..', 'relatorio')
caminho_parte1 = os.path.join(caminho_relatorio, 'parte_1_forca_bruta')
os.makedirs(caminho_parte1, exist_ok=True)

arquivo_grafico = os.path.join(caminho_parte1, '01_crescimento_temporal.png')
plt.savefig(arquivo_grafico, dpi=300, bbox_inches='tight')
print(f"\n✓ Gráfico salvo em: {arquivo_grafico}")

plt.show()

# Salvar resultados em CSV
arquivo_csv_parte1 = os.path.join(caminho_parte1, '02_resultados_forca_bruta.csv')
resultados_forca_bruta.to_csv(arquivo_csv_parte1, index=False, float_format='%.6f')
print(f"✓ Resultados CSV salvo em: {arquivo_csv_parte1}")

print("\nGráfico de escalabilidade exibido e salvo.")

In [ ]:

# Criar estrutura de pastas para salvar resultados
import os

# Criar diretório de relatório
caminho_relatorio = os.path.join('..', 'relatorio')
caminho_parte1 = os.path.join(caminho_relatorio, 'parte_1_forca_bruta')
caminho_parte2 = os.path.join(caminho_relatorio, 'parte_2_heuristica')

os.makedirs(caminho_parte1, exist_ok=True)
os.makedirs(caminho_parte2, exist_ok=True)

print(f"✓ Pastas criadas:")
print(f"  - {caminho_relatorio}/")
print(f"    - {caminho_parte1}/")
print(f"    - {caminho_parte2}/")


---
## Parte 2: Heurística de Coloração

1. Implementar uma heurística para encontrar uma solução para o problema de coloração de vértices
de um grafo. A heurística fica a sua escolha.
2. Aplicar o método implementado no item anterior em cinco instâncias do problema disponíveis no
moodle.
a. 450 vértices e 8260 arestas
b. 864 vértices e 18707 arestas
c. 1000 vértices e 14378 arestas
d. 1916 vértices e 12506 arestas
e. 4730 Vértices e 286722 arestas
3. Verificar o número de cores de cada instância calculada pela sua heurística.
Obs: as instâncias foram retiradas do site Vertex Coloring, está a resposta para cada instância. São
arquivos texto onde as linhas iniciadas com e indicam as adjacências existentes. Seu programa deve
ler cada arquivo, armazenar o grafo e calcular o número de cores “mínima”;

Para instâncias grandes, é impraticável usar força bruta. Utilizaremos uma **heurística gulosa (Greedy Coloring)**.

### Algoritmo Greedy Coloring

O algoritmo greedy funciona da seguinte forma:

1. Ordena os vértices (pode ser em ordem arbitrária ou por grau decrescente)
2. Para cada vértice, atribui a **menor cor disponível** que não foi usada por seus vizinhos coloridos
3. A ordem dos vértices pode impactar a qualidade da solução

### Estratégias de Ordenação

- **Ordem Arbitrária**: Simples, mas pode gerar soluções subótimas
- **Grau Decrescente (DSATUR)**: Prioriza vértices com mais vizinhos, geralmente produz melhores resultados
- **Welsh-Powell**: Variação do DSATUR mais sofisticada

### Complexidade

- **Tempo**: $O(n^2)$ onde $n$ é o número de vértices
- **Espaço**: $O(n)$
- Viável para grafos com milhares de vértices

In [ ]:
def coloracao_greedy(grafo, ordenacao='grau_decrescente'):
    """
    Aplica o algoritmo greedy para coloração de vértices.
    
    Args:
        grafo: Grafo NetworkX
        ordenacao: 'arbitraria' ou 'grau_decrescente'
    
    Returns:
        (número de cores usado, dicionário de coloração)
    """
    # Definir ordem dos vértices
    if ordenacao == 'grau_decrescente':
        # Ordenar por grau decrescente (Welsh-Powell)
        vertices = sorted(grafo.nodes(), key=lambda n: grafo.degree(n), reverse=True)
    else:
        vertices = list(grafo.nodes())
    
    # Inicializar dicionário de coloração
    coloracao = {}
    
    for vertice in vertices:
        # Encontrar cores dos vizinhos já coloridos
        cores_vizinhos = set()
        for vizinho in grafo.neighbors(vertice):
            if vizinho in coloracao:
                cores_vizinhos.add(coloracao[vizinho])
        
        # Encontrar a menor cor disponível
        cor = 0
        while cor in cores_vizinhos:
            cor += 1
        
        coloracao[vertice] = cor
    
    # Calcular número de cores usado
    num_cores = max(coloracao.values()) + 1 if coloracao else 0
    
    return num_cores, coloracao


def coloracao_greedy_dsatur(grafo):
    """
    Aplica o algoritmo DSATUR (Degree of Saturation) para coloração.
    Mais sofisticado que greedy simples.
    
    Args:
        grafo: Grafo NetworkX
    
    Returns:
        (número de cores usado, dicionário de coloração)
    """
    coloracao = {}
    nao_coloridos = set(grafo.nodes())
    
    while nao_coloridos:
        # Calcular grau de saturação para cada vértice não colorido
        dsatur = {}
        for v in nao_coloridos:
            cores_vizinhos = set()
            for u in grafo.neighbors(v):
                if u in coloracao:
                    cores_vizinhos.add(coloracao[u])
            dsatur[v] = (len(cores_vizinhos), -grafo.degree(v))
        
        # Selecionar vértice com maior DSATUR
        v = max(dsatur, key=dsatur.get)
        
        # Encontrar cores dos vizinhos
        cores_vizinhos = set()
        for u in grafo.neighbors(v):
            if u in coloracao:
                cores_vizinhos.add(coloracao[u])
        
        # Encontrar menor cor disponível
        cor = 0
        while cor in cores_vizinhos:
            cor += 1
        
        coloracao[v] = cor
        nao_coloridos.remove(v)
    
    num_cores = max(coloracao.values()) + 1 if coloracao else 0
    return num_cores, coloracao


print("Funções de heurística de coloração definidas.")

### Leitura de Arquivos de Instâncias

As instâncias do Moodle estão em formato de arquivo texto onde linhas iniciadas com 'e' indicam adjacências.

In [ ]:
def ler_arquivo_coloracao(caminho):
    """
    Lê um arquivo de instância de coloração de vértices.
    Formato: linhas começando com 'e' indicam arestas (u v)
    
    Args:
        caminho: Caminho do arquivo
    
    Returns:
        Grafo NetworkX
    """
    G = nx.Graph()
    
    with open(caminho, 'r') as f:
        for linha in f:
            linha = linha.strip()
            
            # Ignorar linhas vazias ou comentários
            if not linha or linha.startswith('c'):
                continue
            
            # Processar linhas que especificam o grafo
            if linha.startswith('p'):
                partes = linha.split()
                n_vertices = int(partes[2])
                n_arestas = int(partes[3])
                G.add_nodes_from(range(1, n_vertices + 1))
            
            # Processar arestas
            elif linha.startswith('e'):
                partes = linha.split()
                u = int(partes[1])
                v = int(partes[2])
                G.add_edge(u, v)
    
    return G


print("Função de leitura de arquivos definida.")

### Aplicação em Instâncias do Moodle

Vamos aplicar a heurística em 5 instâncias grandes fornecidas no Moodle.

**Nota:** Certifique-se de que os arquivos estão no diretório apropriado antes de executar.

In [ ]:
# Exemplo: Testar com instâncias fictícias (simulação)
# Substitua os caminhos pelos caminhos reais das instâncias

# Criar instâncias de teste simuladas
def criar_instancias_teste():
    """
    Cria instâncias de teste com tamanhos similares às do Moodle.
    """
    instancias = {}
    
    # Instância a: 450 vértices, ~8260 arestas
    instancias['a'] = nx.gnp_random_graph(450, 0.08, seed=42)
    
    # Instância b: 864 vértices, ~18707 arestas
    instancias['b'] = nx.gnp_random_graph(864, 0.05, seed=43)
    
    # Instância c: 1000 vértices, ~14378 arestas
    instancias['c'] = nx.gnp_random_graph(1000, 0.03, seed=44)
    
    # Instância d: 1916 vértices, ~12506 arestas
    instancias['d'] = nx.gnp_random_graph(1916, 0.0065, seed=45)
    
    # Instância e: 4730 vértices, ~286722 arestas
    instancias['e'] = nx.gnp_random_graph(4730, 0.02, seed=46)
    
    return instancias


# Testar heurísticas em instâncias
instancias = criar_instancias_teste()

resultados_heuristica = []

for nome, G in instancias.items():
    print(f"\n=== Instância {nome} ===")
    print(f"Vértices: {G.number_of_nodes()}")
    print(f"Arestas: {G.number_of_edges()}")
    
    # Greedy simples
    start = time.time()
    cores_greedy, col_greedy = coloracao_greedy(G, 'grau_decrescente')
    tempo_greedy = time.time() - start
    
    # DSATUR
    start = time.time()
    cores_dsatur, col_dsatur = coloracao_greedy_dsatur(G)
    tempo_dsatur = time.time() - start
    
    print(f"\nGreedy (Grau Decrescente):")
    print(f"  Cores: {cores_greedy}")
    print(f"  Tempo: {tempo_greedy:.4f}s")
    
    print(f"\nDSATUR:")
    print(f"  Cores: {cores_dsatur}")
    print(f"  Tempo: {tempo_dsatur:.4f}s")
    
    resultados_heuristica.append({
        'Instância': nome,
        'Vértices': G.number_of_nodes(),
        'Arestas': G.number_of_edges(),
        'Cores (Greedy)': cores_greedy,
        'Cores (DSATUR)': cores_dsatur,
        'Tempo Greedy (s)': tempo_greedy,
        'Tempo DSATUR (s)': tempo_dsatur
    })

resultados_heuristica_df = pd.DataFrame(resultados_heuristica)

In [ ]:
# Exibir tabela de resultados
print("\n" + "="*100)
print("RESUMO DOS RESULTADOS - HEURÍSTICAS")
print("="*100)
print(resultados_heuristica_df.to_string(index=False))
print("="*100)

In [ ]:
# Visualizar número de cores para cada instância
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Comparação de cores
x = np.arange(len(resultados_heuristica_df))
width = 0.35

axes[0].bar(x - width/2, resultados_heuristica_df['Cores (Greedy)'], 
             width, label='Greedy', color='steelblue')
axes[0].bar(x + width/2, resultados_heuristica_df['Cores (DSATUR)'], 
             width, label='DSATUR', color='coral')
axes[0].set_xlabel('Instância', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Número de Cores', fontsize=12, fontweight='bold')
axes[0].set_title('Comparação de Cores: Greedy vs DSATUR', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(resultados_heuristica_df['Instância'])
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Gráfico 2: Tempo de execução
axes[1].plot(resultados_heuristica_df['Instância'], 
              resultados_heuristica_df['Tempo Greedy (s)'], 
              'o-', linewidth=2, markersize=8, label='Greedy', color='steelblue')
axes[1].plot(resultados_heuristica_df['Instância'], 
              resultados_heuristica_df['Tempo DSATUR (s)'], 
              's-', linewidth=2, markersize=8, label='DSATUR', color='coral')
axes[1].set_xlabel('Instância', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Tempo (segundos)', fontsize=12, fontweight='bold')
axes[1].set_title('Tempo de Execução das Heurísticas', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

# Salvar gráfico
caminho_relatorio = os.path.join('..', 'relatorio')
caminho_parte2 = os.path.join(caminho_relatorio, 'parte_2_heuristica')
os.makedirs(caminho_parte2, exist_ok=True)

arquivo_grafico_heur = os.path.join(caminho_parte2, '01_comparacao_heuristicas.png')
plt.savefig(arquivo_grafico_heur, dpi=300, bbox_inches='tight')
print(f"✓ Gráfico salvo em: {arquivo_grafico_heur}")

plt.show()

# Salvar resultados em CSV
arquivo_csv_parte2 = os.path.join(caminho_parte2, '02_resultados_heuristicas.csv')
resultados_heuristica_df.to_csv(arquivo_csv_parte2, index=False, float_format='%.6f')
print(f"✓ Resultados CSV salvo em: {arquivo_csv_parte2}")

---

## Análise Comparativa

### Resultados da Força Bruta

- **Vantagens**:
  - Garante solução **ótima**
  - Prova matematicamente que encontrou o número cromático exato

- **Desvantagens**:
  - Crescimento **exponencial** do tempo
  - Viável apenas para grafos pequenos (até ~15 vértices)
  - Impraticável para instâncias reais grandes

### Resultados da Heurística

- **Vantagens**:
  - **Linear** ou quadrático em tempo (muito mais rápido)
  - Viável para grafos com milhares de vértices
  - Aplica-se a problemas do mundo real

- **Desvantagens**:
  - Solução **não é garantida ótima**
  - Qualidade depende da heurística e da ordenação dos vértices
  - Pode precisar de ajustes para diferentes tipos de grafos

### Comparação DSATUR vs Greedy

- **DSATUR** geralmente produz **melhores resultados** que greedy simples
- DSATUR leva em consideração o **grau de saturação** dos vértices
- Trade-off: DSATUR é um pouco mais lento que greedy simples

In [ ]:
# Análise estatística
print("\n=== ANÁLISE ESTATÍSTICA ===")
print(f"\nDiferença média entre Greedy e DSATUR:")
diferenca = resultados_heuristica_df['Cores (Greedy)'] - resultados_heuristica_df['Cores (DSATUR)']
print(f"  Média: {diferenca.mean():.2f} cores")
print(f"  Máximo: {diferenca.max()} cores")
print(f"  Mínimo: {diferenca.min()} cores")

print(f"\nRazão de tempo (DSATUR / Greedy):")
razao_tempo = resultados_heuristica_df['Tempo DSATUR (s)'] / resultados_heuristica_df['Tempo Greedy (s)']
print(f"  Média: {razao_tempo.mean():.2f}x mais lento")

# Criar dataframe com análise estatística
analise_stats = pd.DataFrame({
    'Métrica': [
        'Diferença Média Cores',
        'Diferença Máxima Cores',
        'Diferença Mínima Cores',
        'Razão Tempo Média (DSATUR/Greedy)'
    ],
    'Valor': [
        f"{diferenca.mean():.2f}",
        f"{diferenca.max()}",
        f"{diferenca.min()}",
        f"{razao_tempo.mean():.2f}x"
    ]
})

# Salvar análise estatística
caminho_relatorio = os.path.join('..', 'relatorio')
caminho_parte2 = os.path.join(caminho_relatorio, 'parte_2_heuristica')
arquivo_stats = os.path.join(caminho_parte2, '03_analise_estatistica.csv')
analise_stats.to_csv(arquivo_stats, index=False)
print(f"\n✓ Análise estatística salva em: {arquivo_stats}")

print("\n" + "="*60)
print(analise_stats.to_string(index=False))
print("="*60)

---

## Teste com Grafo Específico

Exemplo de coloração completa em um grafo menor para visualizar a solução.

In [ ]:
# Criar um grafo de teste
G_teste = nx.Graph()
G_teste.add_edges_from([
    (1, 2), (1, 3), (1, 4),
    (2, 3), (2, 5),
    (3, 4), (3, 5),
    (4, 5), (4, 6),
    (5, 6)
])

# Testar força bruta
print("Grafo de teste:")
print(f"Vértices: {G_teste.nodes()}")
print(f"Arestas: {G_teste.edges()}")

chi_bruta, col_bruta = numero_cromatico_forca_bruta(G_teste)
print(f"\nForça Bruta: χ = {chi_bruta}")
print(f"Coloração: {col_bruta}")

chi_greedy, col_greedy = coloracao_greedy(G_teste, 'grau_decrescente')
print(f"\nGreedy: χ = {chi_greedy}")
print(f"Coloração: {col_greedy}")

chi_dsatur, col_dsatur = coloracao_greedy_dsatur(G_teste)
print(f"\nDSATUR: χ = {chi_dsatur}")
print(f"Coloração: {col_dsatur}")

In [ ]:
# Visualizar o grafo colorido
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

pos = nx.spring_layout(G_teste, seed=42)

# Cores para visualização
cores_mapa = ['red', 'blue', 'green', 'yellow', 'purple', 'orange']

# Força Bruta
cores_node_bruta = [cores_mapa[col_bruta[v]] for v in G_teste.nodes()]
nx.draw_networkx_nodes(G_teste, pos, node_color=cores_node_bruta, 
                       node_size=1000, ax=axes[0])
nx.draw_networkx_labels(G_teste, pos, font_size=12, font_weight='bold', ax=axes[0])
nx.draw_networkx_edges(G_teste, pos, width=2, ax=axes[0])
axes[0].set_title(f'Força Bruta: χ = {chi_bruta}', fontsize=13, fontweight='bold')
axes[0].axis('off')

# Greedy
cores_node_greedy = [cores_mapa[col_greedy[v]] for v in G_teste.nodes()]
nx.draw_networkx_nodes(G_teste, pos, node_color=cores_node_greedy, 
                       node_size=1000, ax=axes[1])
nx.draw_networkx_labels(G_teste, pos, font_size=12, font_weight='bold', ax=axes[1])
nx.draw_networkx_edges(G_teste, pos, width=2, ax=axes[1])
axes[1].set_title(f'Greedy: χ = {chi_greedy}', fontsize=13, fontweight='bold')
axes[1].axis('off')

# DSATUR
cores_node_dsatur = [cores_mapa[col_dsatur[v]] for v in G_teste.nodes()]
nx.draw_networkx_nodes(G_teste, pos, node_color=cores_node_dsatur, 
                       node_size=1000, ax=axes[2])
nx.draw_networkx_labels(G_teste, pos, font_size=12, font_weight='bold', ax=axes[2])
nx.draw_networkx_edges(G_teste, pos, width=2, ax=axes[2])
axes[2].set_title(f'DSATUR: χ = {chi_dsatur}', fontsize=13, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()

# Salvar gráfico
caminho_relatorio = os.path.join('..', 'relatorio')
caminho_parte2 = os.path.join(caminho_relatorio, 'parte_2_heuristica')
os.makedirs(caminho_parte2, exist_ok=True)

arquivo_grafico_teste = os.path.join(caminho_parte2, '04_grafo_teste_colorido.png')
plt.savefig(arquivo_grafico_teste, dpi=300, bbox_inches='tight')
print(f"✓ Gráfico do grafo teste salvo em: {arquivo_grafico_teste}")

plt.show()

In [ ]:

# Gerar relatório resumido
print("\n" + "="*80)
print("RELATÓRIO FINAL - COLORAÇÃO DE VÉRTICES")
print("="*80)

caminho_relatorio = os.path.join('..', 'relatorio')
caminho_parte1 = os.path.join(caminho_relatorio, 'parte_1_forca_bruta')
caminho_parte2 = os.path.join(caminho_relatorio, 'parte_2_heuristica')

print(f"\n📁 ESTRUTURA DE DIRETÓRIOS:")
print(f"\nrelatorio/")
print(f"├── parte_1_forca_bruta/")
print(f"│   ├── 01_crescimento_temporal.png")
print(f"│   └── 02_resultados_forca_bruta.csv")
print(f"└── parte_2_heuristica/")
print(f"    ├── 01_comparacao_heuristicas.png")
print(f"    ├── 02_resultados_heuristicas.csv")
print(f"    ├── 03_analise_estatistica.csv")
print(f"    └── 04_grafo_teste_colorido.png")

print(f"\n📊 RESUMO PARTE 1 - FORÇA BRUTA:")
print(f"   Instâncias testadas: {len(resultados_forca_bruta)}")
print(f"   Intervalo de tamanho: {resultados_forca_bruta['Tamanho'].min()} a {resultados_forca_bruta['Tamanho'].max()} vértices")
print(f"   Tempo mínimo: {resultados_forca_bruta['Tempo Médio (s)'].min():.6f}s")
print(f"   Tempo máximo: {resultados_forca_bruta['Tempo Médio (s)'].max():.4f}s")

print(f"\n📊 RESUMO PARTE 2 - HEURÍSTICAS:")
print(f"   Instâncias testadas: {len(resultados_heuristica_df)}")
print(f"\n   GREEDY (Grau Decrescente):")
print(f"   - Cores mín/máx: {resultados_heuristica_df['Cores (Greedy)'].min()}/{resultados_heuristica_df['Cores (Greedy)'].max()}")
print(f"   - Tempo total: {resultados_heuristica_df['Tempo Greedy (s)'].sum():.4f}s")
print(f"\n   DSATUR:")
print(f"   - Cores mín/máx: {resultados_heuristica_df['Cores (DSATUR)'].min()}/{resultados_heuristica_df['Cores (DSATUR)'].max()}")
print(f"   - Tempo total: {resultados_heuristica_df['Tempo DSATUR (s)'].sum():.4f}s")

print(f"\n✅ Todos os arquivos foram salvos com sucesso!")
print("="*80)
